# Tutorial 1: Basics and first contact with Inspect AI

Welcome to the first tutorial in our AI Safety Evaluations course.

**What you'll learn:**

- Connect Inspect AI to a language model (locally via Ollama, or via cloud API)
- Run your first evaluation
- Understand how tasks are structured: dataset → solver → scorer
- View and analyze results with `inspect view`
- Create single choice and multiple choice benchmarks
- Analyzing position bias in multiple-choice tasks

**By the end:** You'll have a working evaluation pipeline and understand how to build your own benchmarks.

---
## Prerequisites: Model setup

> **💡 Inspect AI only needs a model name** — the model itself can come from anywhere.

**In this tutorial, we'll use Ollama and Perplexity, SambaNova as examples**, but you can substitute any provider: OpenAI, Anthropic, Google, local inference servers, or any OpenAI-compatible endpoint.

**Cost note:** Cloud APIs have small free tiers and then charge per token. Local models (Ollama) are completely free. For this course, a local model is sufficient for all assignments.

See [Inspect AI models docs](https://inspect.ai-safety-institute.org.uk/models.html) for the full list.

---
## Part 1: Local environment setup (Ollama)

Running evaluations locally gives you complete control and privacy.

### 1.1. Installing Ollama

**Before running this notebook, install Ollama:**

**macOS:** `brew install ollama` or download from https://ollama.ai/download

**Linux:**
```bash
curl -fsSL https://ollama.ai/install.sh | sh
```

**Windows:** download from https://ollama.ai/download

After installation, start the server and download a model:
```bash
ollama serve
ollama pull llama2        # or use deepseek-r1:1.5b for a smaller option
```

### 1.2. Check Ollama connection

`ollama serve`

In [1]:
import requests

def check_ollama():
    """Check Ollama connection and show installed models."""
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        if response.status_code != 200:
            print("❌ Ollama returned an error")
            return False
            
        models = response.json().get('models', [])
        total_size = sum(m['size'] for m in models)
        
        print("✅ Ollama is running!")
        print(f"\n📊 Installed models: {len(models)} ({total_size / 1e9:.1f} GB total)")
        
        for m in models:
            print(f"   - {m['name']}: {m['size'] / 1e9:.2f} GB")
        
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to Ollama")
        print("   Start it with: ollama serve")

check_ollama()

✅ Ollama is running!

📊 Installed models: 1 (3.8 GB total)
   - llama2:latest: 3.83 GB


---
## Part 2: Basic Inspect setup

### 2.1. Install Inspect AI

## Assignment 1: 'Hello world' in eval

Let's create the simplest possible evaluation!

**To do:** add one more `Sample()` to the dataset.

In [2]:
from inspect_ai import Task, task, eval
from inspect_ai.dataset import Sample
from inspect_ai.scorer import exact, match, model_graded_fact, choice, pattern
from inspect_ai.solver import (
    generate, system_message, chain_of_thought, 
    prompt_template, multiple_choice
)

In [3]:
@task
def hello_model():
    """Test your model setup with simple questions."""
    return Task(
        dataset=[
            Sample(
                input="Say 'Hello world!' and nothing else.",
                target="Hello world!"
            ),
            Sample(
                input="2+2=",
                target="4"
            ),
            Sample(
                input="What is the surname of Sheldon from The Big Bang Theory?",
                target="Cooper"
            ),
            
            Sample(
                input="Напиши слово 'яблоко' наоборот и ничего больше.", # YOUR CODE HERE
                target="околбя" # YOUR CODE HERE
            )
        ],
        solver=[generate()],
        scorer= match(
            location="end",        # where to look for the answer: "begin", "end", "any", "exact"
            ignore_case=True,      # ignore case when comparing
            numeric=False          # treat as numeric comparison (normalizes numbers, different punctuation rules)
        )
    )

**Run the evaluation:**

This will take a minute or two depending on your hardware.

In [4]:
eval(
    hello_model,
    model="ollama/llama2",
    # limit=1  # Uncomment to test with just 1 sample
)

Output()

---
## Part 3: API setup (optional)

If you don't have a GPU or want to test cloud models, you can use API providers.

### 3.1. Perplexity setup

1. Get an API key at https://www.perplexity.ai/settings/api
2. Set it in the cell below

## Sambanova 

Get API KEY https://cloud.sambanova.ai/apis here 

## Local OpenAI-compatible model

In [5]:
import os

%load_ext dotenv
%dotenv /root/.env

os.environ["OPENAI_API_KEY"] = os.environ["LITELLM_API_KEY"]
os.environ["OPENAI_BASE_URL"] = "https://srs-litellm.kontur.host/v1"

In [6]:
eval(
    hello_model,
    model="openai/code-pro",
    # limit=1
)

Output()

---
## Part 4: Viewing results with Inspect view

Every evaluation saves a log file. `inspect view` opens a web UI to explore them.

### 4.1. Launch Inspect view

1. In terminal, from the notebook's folder, run: `inspect view`
2. Open in browser: http://localhost:7575

This will:
1. Show all evaluation logs in an interactive interface
2. Allow you to drill down into individual samples

**Alternative options:**

```bash
# View logs from a specific directory
inspect view --log-dir ./experiment-logs

# Use a different port
inspect view --port 8080
```

**Troubleshooting:**

- If `inspect: command not found` → try `python -m inspect_ai view`
- If the page won't load → check that you're in the correct folder (logs are saved relative to where you run evaluations)

`inspect view --log-dir logs/`

`hostname -I`

In [7]:
from IPython.display import IFrame

IFrame(src="http://10.232.18.235:7575", width=800, height=400)

In [8]:
import inspect_ai
from inspect_ai.log import list_eval_logs, read_eval_log

logs = list_eval_logs("logs/")
for log_path in logs:
    log = read_eval_log(log_path)
    print(f'{log.status}: {list(log.model_dump().keys())}')

success: ['version', 'status', 'eval', 'plan', 'results', 'stats', 'error', 'invalidated', 'log_updates', 'tags', 'metadata', 'samples', 'reductions']
success: ['version', 'status', 'eval', 'plan', 'results', 'stats', 'error', 'invalidated', 'log_updates', 'tags', 'metadata', 'samples', 'reductions']


In [9]:
from pathlib import Path

log_files = list(Path("./logs").glob("*.eval")) if Path("./logs").exists() else []

if not log_files:
    print("❌ No log files found")
    print("   Run at least one eval() in this notebook first")
    print(f"   Current directory: {os.getcwd()}")
else:
    print(f"✅ Found {len(log_files)} log file(s):")
    for f in sorted(log_files)[-5:]:  # show last 5
        print(f"   {f.name}")

✅ Found 2 log file(s):
   2026-03-21T13-09-52+00-00_hello-model_bGn4bBqJDL6wgjKAm9Qio6.eval
   2026-03-21T13-10-57+00-00_hello-model_FTjfSJvw3X2Z2XtZ6fXgL4.eval


### Assignment 2: Explore your logs

In the Inspect view UI you can:

- **See overall accuracy** for each evaluation run
- **Click on individual samples** to see the model's response
- **Compare runs** with different models or parameters
- **Filter by metadata** (e.g., show only "hard" problems)
- **Export results** for further analysis

**Tip:** Keep `inspect view` running in a separate terminal while you work through this notebook. It auto-refreshes when new evaluations complete.

---
## Part 5: Understanding benchmark structure

### 5.1. Task components overview

Every Inspect `Task` consists of:

```
Task {
    dataset: [Sample, Sample, ...],    # data to evaluate on
    solver: [Solver, Solver, ...],     # how to process
    scorer: Scorer,                    # how to score
    **parameters                       
}
```

**Component flow:**
```
Dataset (Samples) → Solver(s) → Model → Scorer → Results
```

### 5.2. Sample structure

A Sample contains input/target pairs with optional metadata:

In [10]:
# Example of a fully-featured Sample
sample_example = Sample(
    input="Question or prompt",
    target="Expected answer",
    id="unique_id",
    choices=["Option A", "Option B", "Option C"],  # For multiple choice
    metadata={
        "category": "math",
        "difficulty": "hard"
    }
)

print("Sample components:")
print(f"  - input: {sample_example.input}")
print(f"  - target: {sample_example.target}")
print(f"  - choices: {sample_example.choices}")
print(f"  - metadata: {sample_example.metadata}")

Sample components:
  - input: Question or prompt
  - target: Expected answer
  - choices: ['Option A', 'Option B', 'Option C']
  - metadata: {'category': 'math', 'difficulty': 'hard'}


---
## Part 6: Understanding solvers

### 6.1. What is a solver?

A **solver** is a function that transforms a **TaskState** (the prompt + conversation history) and optionally calls the model to generate a response.

**Think of solvers as middleware that:**
1. Modifies the prompt (prompt engineering)
2. Calls the model (generation)
3. Processes the response (extraction, critique, etc.)

### 6.2. The solver pipeline

Solvers are chained together in a pipeline:

```
Input Sample
    ↓
[Solver 1: system_message]
    ↓
[Solver 2: prompt_template]
    ↓
[Solver 3: chain_of_thought]
    ↓
[Solver 4: generate]
    ↓
Model Output → Scorer → Final Result
```

Each solver receives the TaskState, modifies it, and passes it to the next solver.

### 6.3. TaskState - the core data structure

Every solver operates on a **TaskState** containing:

```
TaskState {
    messages: list[ChatMessage],  # Conversation history
    output: ModelOutput,          # Final model output
    user_prompt: str,             # Current user prompt
    input_text: str,              # Original input
    metadata: dict,               # Sample metadata
    choices: list[str],           # For multiple choice
    model: ModelName,             # Current model
    sample_id: int | str,         # Sample identifier
}
```

---
## Part 7: Built-in solvers

**system_message**
```python
system_message(
    message: str        # REQUIRED - the system prompt
)
```

**prompt_template**
```python
prompt_template(
    template: str       # REQUIRED - use {prompt} as placeholder
)
```

**chain_of_thought**
```python
chain_of_thought(
    template: str = None   # optional - custom CoT prompt (default: "Let's think step by step")
)
```

**generate**
```python
generate(
    max_tokens: int = None,      # optional - limit response length
    temperature: float = None,   # optional - 0.0 = deterministic, 1.0 = creative
    top_p: float = None,         # optional - nucleus sampling
    stop_seqs: list[str] = None  # optional - stop generation at these strings
)
```

**multiple_choice**
```python
multiple_choice(
    cot: bool = False,              # optional - add chain-of-thought
    multiple_correct: bool = False, # optional - allow multiple answers
    shuffle: bool = False           # optional - randomize choice order
)
```

**Typical pipeline:**
```
system_message → prompt_template → chain_of_thought → generate
```

`multiple_choice()` replaces the entire chain - it handles prompting and generation internally.

**Viewing solver execution:** in `inspect view`, click any sample → messages tab shows each solver's contribution.

### 7.1 system_message()

**Purpose:** prepend a system role message to guide model behavior.

**When to use:**
- establish the model's role or persona
- set global guidelines or constraints
- define the evaluation context


In [11]:
@task
def example_system_message():
    """
    Demonstrates system_message() solver.
    The system prompt tells the model to be concise.
    """
    return Task(
        dataset=[
            Sample(input="What is 15 * 8?", target="120"),
            Sample(input="What is 99 + 1?", target="100"),
        ],
        solver=[
            system_message("You are a calculator. Reply with only the number, nothing else."),
            generate()
        ],
        scorer=match(numeric=True),
    )

# Run and check the Messages tab in inspect view
eval(example_system_message, model="openai/code-pro")

Output()

### 7.2 prompt_template()

**Purpose:** substitute variables into a template to reformat prompts.

**When to use:**
- add specific output format requirements
- include examples or demonstrations
- structure prompts consistently
- add reasoning steps or breakdowns


In [12]:
STEP_BY_STEP_TEMPLATE = '''
Solve this problem step by step:

Problem: {prompt}

Structure:
1. Understand the problem
2. Plan your approach
3. Solve it
4. Final answer format: ANSWER: <value>
'''.strip()

@task
def example_prompt_template():
    """
    Demonstrates prompt_template() solver.
    The template adds structure to the prompt.
    """
    return Task(
        dataset=[
            Sample(input="What is 25 * 4?", target="100"),
            Sample(input="What is 144 / 12?", target="12"),
        ],
        solver=[
            system_message("You are a math tutor."),
            prompt_template(STEP_BY_STEP_TEMPLATE),
            generate()
        ],
        scorer=match(numeric=True),
    )

# Run and see how the template structures the prompt
eval(example_prompt_template, model="openai/code-pro")

Output()

### 7.3 chain_of_thought()

**Purpose:** ask the model to "think step by step" before answering.

**When to use:**
- math and logic problems
- multi-step reasoning tasks
- when you want to see the model's thought process

In [13]:
@task
def example_chain_of_thought():
    """
    Demonstrates chain_of_thought() solver.
    Compare accuracy with and without CoT in inspect view.
    """
    return Task(
        dataset=[
            Sample(
                input="If Alice has 3 apples and Bob gives her 2 more, how many does she have?",
                target="5"
            ),
            Sample(
                input="A train travels 100 km in 2 hours. At this rate, how far in 5 hours?",
                target="250"
            ),
        ],
        solver=[
            system_message("Solve the problem. End with: ANSWER: <number>"),
            chain_of_thought(),
            generate()
        ],
        scorer=match(numeric=True),
    )

eval(example_chain_of_thought, model="openai/code-pro")

Output()

### 7.5. multiple_choice()

Special solver for A/B/C/D questions. Handles formatting and answer extraction automatically.

**When to use:**
- multiple choice questions (use instead of generate)
- must have letter target: "A", "B", "C", etc.

**Note:** When using `multiple_choice()`, use `choice()` as the scorer.

In [14]:
@task
def example_multiple_choice_with_cot():
    """
    Demonstrates multiple_choice(cot=True).
    Model reasons before selecting an answer.
    """
    return Task(
        dataset=[
            Sample(
                input="Light travels faster than sound. If you see lightning and hear thunder 3 seconds later, approximately how far away was the strike?",
                choices=["100 meters", "1 kilometer", "3 kilometers", "10 kilometers"],
                target="B"  # ~1 km (sound travels ~340 m/s)
            ),
        ],
        solver=multiple_choice(cot=True),
        scorer=choice(),
    )

eval(example_multiple_choice_with_cot, model="openai/code-pro")

Output()

### 7.6 Other solvers

**self_critique()** - Have model refine its own answer
```python
solver=[generate(), self_critique()]
```

**use_tools()** - Enable tool/function calling
```python
solver=[use_tools(calculator()), generate()]
```

---
## Part 8: Single choice tasks

Single choice tasks present the model with limited options to select from.

### 8.1. Simple yes/no classification

The simplest single choice - binary classification:

In [15]:
@task
def yes_no_classification():
    return Task(
        dataset=[
            Sample(
                input="Is Python a programming language?",
                target="Yes"
            ),
            Sample(
                input="Is water dry?",
                target="No"
            ),
            Sample(
                input="Is the Earth round?",
                target="Yes"
            ),
        ],
        solver=[
            system_message("Answer 'Yes' or 'No'. Be concise."),
            generate()
        ],
        scorer=exact(),
    )

eval(
    yes_no_classification,
    model="openai/code-pro"
)

Output()

### 8.2. Multi-class classification

In multi-class classification, the model must choose from 3+ categories. This is common for:
- sentiment analysis (positive / negative / neutral)
- topic classification (sports / politics / tech / ...)
- intent detection (question / command / statement)

---

### Task 2: Build a sentiment classifier

**Your goal:** Create a sentiment classification task with at least 4 samples.

**Note:**
- `system_message` defines the classes and output format
- `target` must exactly match one of your class labels

In [16]:
@task
def sentiment_classification():
    return Task(
        dataset=[
            Sample(
                input="I love this product! It's amazing.",
                target="positive"
            ),
            Sample(
                input="This movie was terrible and boring.",
                target="negative"
            ),
            Sample(
                input="The weather today is okay, not great but not bad.",
                target="neutral"
            ),
            Sample(
                input="I'm so happy with the service!",
                target="positive"
            ),
        ],
        solver=[
            system_message("Classify the sentiment of the following text as positive, negative, or neutral. Respond with only the label."),
            generate()
        ],
        scorer=exact()
    )

eval(
    sentiment_classification,
    model="openai/code-pro"
)

Output()

In [17]:
from inspect_ai.log import read_eval_log

log = read_eval_log("logs/2026-03-21T13-15-14+00-00_sentiment-classification_mQyzfLd9TDcwDixwrEETtq.eval")

for sample in log.samples:
    print(f"Input: {sample.input}")
    print(f"Target: {sample.target}")
    print(f"Output: {sample.output}")
    print(f"Score: {sample.score}")
    print("-" * 40)

Input: I love this product! It's amazing.
Target: positive
Output: model='qwen3-coder-30b-a3b-instruct-fp8' choices=[ChatCompletionChoice(message=ChatMessageAssistant(id='AQrYc5Tg9LfvPtTKsBbpK9', content='positive', source='generate', metadata=None, role='assistant', tool_calls=None, model='qwen3-coder-30b-a3b-instruct-fp8'), stop_reason='stop', logprobs=None)] completion='positive' usage=ModelUsage(input_tokens=44, output_tokens=2, total_tokens=46, input_tokens_cache_write=None, input_tokens_cache_read=None, reasoning_tokens=None, total_cost=None) time=0.12506468780338764 metadata=None error=None


[03/21/26 16:15:47] WARNING  The 'score' field is deprecated. Access sample scores through 'scores'   ]8;id=491757;file:///root/.local/lib/python3.10/site-packages/inspect_ai/_util/logger.py\logger.py]8;;\:]8;id=300667;file:///root/.local/lib/python3.10/site-packages/inspect_ai/_util/logger.py#220\220]8;;\
                             instead.                                                                              

Score: value='C' answer='positive' explanation=None metadata=None history=[]
----------------------------------------
Input: This movie was terrible and boring.
Target: negative
Output: model='qwen3-coder-30b-a3b-instruct-fp8' choices=[ChatCompletionChoice(message=ChatMessageAssistant(id='PfzcUid7n2tc9ZEoj3M5TV', content='negative', source='generate', metadata=None, role='assistant', tool_calls=None, model='qwen3-coder-30b-a3b-instruct-fp8'), stop_reason='stop', logprobs=None)] completion='negative' usage=ModelUsage(input_tokens=42, output_tokens=2, total_tokens=44, input_tokens_cache_write=None, input_tokens_cache_read=None, reasoning_tokens=None, total_cost=None) time=0.1303513515740633 metadata=None error=None
Score: value='C' answer='negative' explanation=None metadata=None history=[]
----------------------------------------
Input: The weather today is okay, not great but not bad.
Target: neutral
Output: model='qwen3-coder-30b-a3b-instruct-fp8' choices=[ChatCompletionChoice(message

### 8.3. Single choice with explanation

Collect both choice and reasoning:

In [18]:
@task
def choice_with_reasoning():
    PROMPT = '''
Classify as True or False:

Statement: {prompt}

Provide:
1. REASONING: [Your explanation]
2. ANSWER: [True or False]
    '''.strip()

    return Task(
        dataset=[
            Sample(
                input="The Earth is flat.",
                target="False"
            ),
            Sample(
                input="Water boils at 100°C at sea level.",
                target="True"
            ),
        ],
        solver=[
            chain_of_thought(),
            prompt_template(PROMPT),
            generate()
        ],
        scorer=pattern(r'ANSWER:\s*(True|False)'),
    )

eval(choice_with_reasoning, model='openai/code-pro')

Output()

In [19]:
from inspect_ai.log import read_eval_log

log = read_eval_log("logs/2026-03-21T13-16-51+00-00_choice-with-reasoning_TDFbqUFqZM2ayKQusJa9xp.eval")

for sample in log.samples:
    print(f"Input: {sample.input}")
    print(f"Target: {sample.target}")
    print(f"Output: {sample.output}")
    print(f"Score: {sample.score}")
    print("-" * 40)

Input: The Earth is flat.
Target: False
Output: model='qwen3-coder-30b-a3b-instruct-fp8' choices=[ChatCompletionChoice(message=ChatMessageAssistant(id='CkfhL7iaUcnZCBZKo65Vaz', content='1. REASONING: Let me analyze this statement step by step.\n\nFirst, I need to consider what evidence exists about the Earth\'s shape:\n- The Earth is an oblate spheroid (slightly flattened sphere) that rotates on its axis\n- We have extensive scientific evidence from multiple sources including:\n  - Satellite imagery and space observations\n  - Ships disappearing hull-first over the horizon\n  - Different star constellations visible from different latitudes\n  - Time zones and varying daylight hours\n  - Gravity measurements\n  - Flight paths and navigation systems\n  - Photos from space showing the Earth\'s spherical shape\n\nSecond, I should consider what "flat" would mean:\n- A flat Earth would require the entire surface to be planar\n- This contradicts basic physics and observable phenomena\n- It wo

---
## Part 9: Multiple choice tasks

### 9.1. Understanding multiple choice in Inspect

Key rules:
- `choices`: list of answer options (no letters — they're added automatically)
- `target`: letter of correct answer ("A", "B", "C", or "D")
- Use `multiple_choice()` solver + `choice()` scorer

### 9.2. Multiple choice with metadata

Metadata lets you filter and analyze results in `inspect view`.

In [20]:
@task
def mc_with_metadata():
    return Task(
        dataset=[
            Sample(
                input="Capital of Japan?",
                choices=["Seoul", "Tokyo", "Bangkok", "Beijing"],
                target="B",
                metadata={
                    "difficulty": "easy",
                    "category": "geography"
                }
            ),
            Sample(
                input="What is the Heisenberg Uncertainty Principle?",
                choices=[
                    "Cannot know both position and momentum precisely",
                    "Energy cannot be created or destroyed",
                    "All matter has wave-particle duality",
                    "Time always moves forward"
                ],
                target="A",
                metadata={
                    "difficulty": "hard",
                    "category": "physics"
                }
            ),
        ],
        solver=multiple_choice(),
        scorer=choice(),
    )

# Run and check results in inspect view - filter by metadata!
eval(mc_with_metadata, model='openai/code-pro')

Output()

In [21]:
log = read_eval_log("logs/2026-03-21T13-17-42+00-00_mc-with-metadata_Z3UWJPsPAFLWFcpak42Ndq.eval")

for sample in log.samples:
    print(f"Input: {sample.input}")
    print(f"Target: {sample.target}")
    print(f"Output: {sample.output}")
    print(f"Score: {sample.score}")
    print("-" * 40)

Input: Capital of Japan?
Target: B
Output: model='qwen3-coder-30b-a3b-instruct-fp8' choices=[ChatCompletionChoice(message=ChatMessageAssistant(id='NMABSUgdEkYXtrPM4ENwPp', content='ANSWER: B', source='generate', metadata=None, role='assistant', tool_calls=None, model='qwen3-coder-30b-a3b-instruct-fp8'), stop_reason='stop', logprobs=None)] completion='ANSWER: B' usage=ModelUsage(input_tokens=69, output_tokens=5, total_tokens=74, input_tokens_cache_write=None, input_tokens_cache_read=None, reasoning_tokens=None, total_cost=None) time=0.12972859665751457 metadata=None error=None
Score: value='C' answer='B' explanation='ANSWER: B' metadata=None history=[]
----------------------------------------
Input: What is the Heisenberg Uncertainty Principle?
Target: A
Output: model='qwen3-coder-30b-a3b-instruct-fp8' choices=[ChatCompletionChoice(message=ChatMessageAssistant(id='EhjRZEwK34AhdQ5nDzBqpi', content='ANSWER: A', source='generate', metadata=None, role='assistant', tool_calls=None, model='qw

### 9.6. Multiple correct Answers

When multiple answers are valid:

In [22]:
@task
def mc_multiple_correct():
    return Task(
        dataset=[
            Sample(
                input="Which are programming languages?",
                choices=["Python", "HTML", "JavaScript", "CSS"],
                target=["A", "C"]  # Python, JavaScript
            ),
            Sample(
                input="Which continents border the Atlantic Ocean?",
                choices=["Africa", "Asia", "Europe", "South America"],
                target=["A", "C", "D"]  # Africa, Europe, South America
            ),
        ],
        solver=[
            system_message("Select ALL correct answers. You may choose multiple options."),
            multiple_choice(multiple_correct=True)
        ],
        scorer=choice(),
    )

eval(mc_multiple_correct, model='openai/code-pro')

Output()

---
## Part 10: Composing solvers together

## Quick reference

| Task type | Solvers | Scorer |
|-----------|---------|--------|
| Simple Q&A | `system_message() + generate()` | `match()` |
| Reasoning | `chain_of_thought() + generate()` | `match()` |
| Structured output | `prompt_template() + generate()` | `pattern()` |
| Classification | `system_message() + generate()` | `exact()` |
| Multiple choice | `multiple_choice()` | `choice()` |
| MC + reasoning | `multiple_choice(cot=True)` | `choice()` |

---
## Assignment 3: Analyzing position bias in multiple choice

Language models can develop **position bias** - a tendency to favor certain answer positions (like more often picking "A" or "C") regardless of content.

In this assignment, you will:
1. Generate a set of simple math questions in multiple-choice format
2. Create two versions of the dataset:
   - **Biased:** correct answer is always position A
   - **Unbiased:** correct answer position is randomized
3. Run evaluations on both and compare results
4. Analyze whether the model shows position bias

⚠️ **Note on methodology:** this is a minimal experiment to get you started. Comparing "all-A" vs "randomized" datasets is a quick sanity check, but it's not the most rigorous way to measure position bias.

Feel free to extend the assignment if you want a deeper analysis!

In [23]:
import random
from inspect_ai import Task, task, eval
from inspect_ai.dataset import Sample
from inspect_ai.scorer import choice
from inspect_ai.solver import multiple_choice, system_message

# For reproducibility
random.seed(42)

### Step 1: Generate questions

First, create a helper function that generates simple questions with known correct answers.

**Function spec:**
- Input: `n` (number of problems to generate)
- Output: list of tuples `(question_text, correct_answer)`
- Example output: `[("What is 5 + 3?", "8"), ("What is 12 - 4?", "8"), ...]`

**This is just an example.** You can implement your own generator with different content - trivia, vocabulary, geography, etc. Just make sure the model can reasonably answer them.

In [24]:
def generate_questions(n: int) -> list[tuple[str, str]]:
    """
    Generate n simple math problems.
    
    Args:
        n: number of problems to generate
        
    Returns:
        List of (question_text, correct_answer) tuples
    """
    problems = []
    operations = ['+', '-', '*']

    for _ in range(n):
        op = random.choice(operations)

        if op == '+':
            a = random.randint(0, 100)
            b = random.randint(0, 100)
            answer = str(a + b)
            question = f"What is {a} + {b}?"
        elif op == '*':
            a = random.randint(0, 100)
            b = random.randint(0, 100)
            answer = str(a * b)
            question = f"What is {a} * {b}?"
        else:  # subtraction
            a = random.randint(30, 100)
            b = random.randint(1, a)   # b <= a
            answer = str(a - b)
            question = f"What is {a} - {b}?"

        problems.append((question, answer))

    return problems

In [25]:
# ===== TESTS =====
test_questions = generate_questions(5)

assert len(test_questions) == 5, f"Expected 5 questions, got {len(test_questions)}"
assert all(isinstance(q, tuple) and len(q) == 2 for q in test_questions), "Each question must be a tuple of (question_text, answer)"
assert all(isinstance(q[0], str) and isinstance(q[1], str) for q in test_questions), "Both question and answer must be strings"
assert all(len(q[0]) > 0 and len(q[1]) > 0 for q in test_questions), "Question and answer cannot be empty"

print("\nSample output:")
for q, a in test_questions:
    print(f"  {q} → {a}")


Sample output:
  What is 14 * 3? → 42
  What is 35 * 31? → 1085
  What is 17 + 94? → 111
  What is 86 + 94? → 180
  What is 11 * 75? → 825


### Step 2: Create wrong answers (distractors)

For multiple choice, we need plausible wrong answers.

**Function spec:**
- Input: `correct_answer` (int)
- Output: list of 3 wrong answers (ints), all different from correct and from each other

**Tip:** generate distractors close to the correct answer (e.g., ±1, ±2, ±10) to make them plausible.

In [26]:
def generate_distractors(correct: str, n: int = 3) -> list[str]:
    """
    Generate n plausible wrong answers for a numeric correct answer.
    
    Args:
        correct: the correct answer as a string (e.g., "8")
        n: number of distractors to generate (default 3)
        
    Returns:
        List of n distinct wrong answers (as strings)
    """
    distractors = set()
    offsets = range(-10*n, 10*n)

    while len(distractors) < n:
        offset = random.choice(offsets)
        if offset == 0:
            continue

        candidate = int(correct) + offset
        candidate_str = str(candidate)

        if candidate_str not in distractors:
            distractors.add(candidate_str)

    return list(distractors)

In [27]:
# ===== TESTS =====
test_distractors = generate_distractors("10", n=3)

assert len(test_distractors) == 3, f"Expected 3 distractors, got {len(test_distractors)}"
assert all(isinstance(d, str) for d in test_distractors), "All distractors must be strings"
assert "10" not in test_distractors, "Distractors must not include the correct answer"
assert len(set(test_distractors)) == 3, "All distractors must be unique"

print(f"   Distractors for '10': {test_distractors}")

   Distractors for '10': ['7', '-19', '-18']


### Step 3: Create multiple choice samples

Now create a function that converts questions into multiple-choice format.

**Function spec:**
- Input: 
  - `questions` - list of `(question_text, correct_answer)` tuples
  - `correct_position` - where to place correct answer:
    - `None` → randomize position for each question
    - `0` → always position A
    - `1` → always position B
    - `2` → always position C
    - `3` → always position D
- Output:
    - list of `Sample` objects

⚠️ **Note on `Sample` type:**

`Sample` is an Inspect AI class. For multiple choice, you create it like this:
```python
Sample(
    input="What is 2 + 2?",           # question text
    choices=["3", "4", "5", "6"],     # list of options: list[str]
    target="B"                        # letter of correct answer (A/B/C/D)
)
```

In [28]:
def create_samples(
    questions: list[tuple[str, str]], 
    correct_position: int | None = None
) -> list[Sample]:
    """
    Convert questions to multiple-choice Samples.
    
    Args:
        questions: list of (question_text, correct_answer) tuples
        correct_position: 
            None → randomize position (A/B/C/D) for each question
            0 → correct answer always at position A
            1 → correct answer always at position B  
            2 → correct answer always at position C
            3 → correct answer always at position D
            
    Returns:
        List of Sample objects ready for Inspect AI.
        Each Sample has:
            - input: str (the question)
            - choices: list[str] (4 options, no letters)
            - target: str (correct letter: "A", "B", "C", or "D")
    """
    samples = []
    
    for question, correct in questions:
        distractors = generate_distractors(correct)
        
        if correct_position is None:
            options = [correct] + distractors
            random.shuffle(options)
            correct_index = options.index(correct)
            target = chr(ord('A') + correct_index)
        else:
            options = [None] * 4
            options[correct_position] = correct
            dist_idx = 0
            for i in range(4):
                if i != correct_position:
                    options[i] = distractors[dist_idx]
                    dist_idx += 1
            target = chr(ord('A') + correct_position)
        
        samples.append(Sample(input=question, choices=options, target=target))
    
    return samples

In [29]:
# ===== TESTS =====
test_q = [("What is 2 + 2?", "4"), ("What is 10 - 3?", "7"), ("What is 5 + 5?", "10")]
samples_fixed = create_samples(test_q, correct_position=0)
samples_random = create_samples(test_q, correct_position=None)

assert len(samples_fixed) == len(test_q), f"Expected {len(test_q)} samples, got {len(samples_fixed)}"
assert all(hasattr(s, 'input') and hasattr(s, 'choices') and hasattr(s, 'target') for s in samples_fixed), "Each sample must have 'input', 'choices', and 'target' attributes"
assert all(len(s.choices) == 4 for s in samples_fixed), "Each sample must have exactly 4 choices"
assert all(s.target == "A" for s in samples_fixed), "With correct_position=0, all targets should be 'A'"
assert all(s.choices[0] == correct for s, (_, correct) in zip(samples_fixed, test_q)), "With correct_position=0, correct answer should be first in choices"
assert all(s.target in "ABCD" for s in samples_random), "Target must be one of A, B, C, D"

# Check that correct answer is actually at the target position
for s, (_, correct) in zip(samples_random, test_q):
    target_index = "ABCD".index(s.target)
    assert s.choices[target_index] == correct, f"Correct answer '{correct}' should be at position {s.target}, but found '{s.choices[target_index]}'"

### Step 4: Create the tasks

Now wrap your datasets into Inspect Tasks.

**Already provided:** task structure. You just need to call your functions.

In [30]:
@task
def position_bias_task(
    questions: list[tuple[str, int]],
    correct_position: int | None = None
):
    """
    Multiple choice evaluation task.
    
    Args:
        questions: list of (question_text, correct_answer) tuples
        correct_position: None for random, 0-3 for fixed position
    """
    samples = create_samples(questions, correct_position)
    return Task(
        dataset=samples,
        solver =multiple_choice(),
        scorer =choice()
    )

### Step 5: Generate questions and run evaluations

Generate questions once, then run two evaluations:
1. **Biased:** correct answer always at position A
2. **Unbiased:** correct answer position randomized

In [31]:
MODEL = 'openai/code-pro'
N_QUESTIONS = 100

random.seed(42)
questions = generate_questions(N_QUESTIONS)

eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": 0}
)

Output()

In [32]:
eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": None}
)

Output()

In [33]:
log = read_eval_log("logs/2026-03-21T13-20-28+00-00_position-bias-task_EGKjoZeVRdpymhDhBDCavd.eval")

for sample in log.samples[:3]:
    print(f"Input: {sample.input}")
    print(f"Target: {sample.target}")
    print(f"Output: {sample.output}")
    print(f"Score: {sample.score}")
    print("-" * 40)

Input: What is 14 * 3?
Target: A
Output: model='qwen3-coder-30b-a3b-instruct-fp8' choices=[ChatCompletionChoice(message=ChatMessageAssistant(id='aASaySxs9S6bjyUUftyfRv', content='ANSWER: A', source='generate', metadata=None, role='assistant', tool_calls=None, model='qwen3-coder-30b-a3b-instruct-fp8'), stop_reason='stop', logprobs=None)] completion='ANSWER: A' usage=ModelUsage(input_tokens=82, output_tokens=5, total_tokens=87, input_tokens_cache_write=None, input_tokens_cache_read=None, reasoning_tokens=None, total_cost=None) time=0.22396457567811012 metadata=None error=None
Score: value='C' answer='A' explanation='ANSWER: A' metadata=None history=[]
----------------------------------------
Input: What is 35 * 31?
Target: A
Output: model='qwen3-coder-30b-a3b-instruct-fp8' choices=[ChatCompletionChoice(message=ChatMessageAssistant(id='MshGTAGxEYkjuzRjWhQw9d', content='ANSWER: A', source='generate', metadata=None, role='assistant', tool_calls=None, model='qwen3-coder-30b-a3b-instruct-fp8'

In [34]:
log = read_eval_log("logs/2026-03-21T13-20-37+00-00_position-bias-task_WJ58htXWYpCQciWY3Pvq9U.eval")

for sample in log.samples[:3]:
    print(f"Input: {sample.input}")
    print(f"Target: {sample.target}")
    print(f"Output: {sample.output}")
    print(f"Score: {sample.score}")
    print("-" * 40)

Input: What is 14 * 3?
Target: D
Output: model='qwen3-coder-30b-a3b-instruct-fp8' choices=[ChatCompletionChoice(message=ChatMessageAssistant(id='QCxo5v5XDW6x3YBxmGakLn', content='ANSWER: D', source='generate', metadata=None, role='assistant', tool_calls=None, model='qwen3-coder-30b-a3b-instruct-fp8'), stop_reason='stop', logprobs=None)] completion='ANSWER: D' usage=ModelUsage(input_tokens=82, output_tokens=5, total_tokens=87, input_tokens_cache_write=None, input_tokens_cache_read=None, reasoning_tokens=None, total_cost=None) time=0.20318572595715523 metadata=None error=None
Score: value='C' answer='D' explanation='ANSWER: D' metadata=None history=[]
----------------------------------------
Input: What is 35 * 31?
Target: A
Output: model='qwen3-coder-30b-a3b-instruct-fp8' choices=[ChatCompletionChoice(message=ChatMessageAssistant(id='grfKSHL3Awmw5G7Fd7dpqz', content='I need to calculate 35 × 31.\n\nI can break this down using the distributive property:\n35 × 31 = 35 × (30 + 1) = (35 × 3

### Step 7: Analyze your results

Open `inspect view` and examine both evaluation runs.

1. **Accuracy comparison:**
   - Biased task accuracy: ____%
   - Unbiased task accuracy: ____%
  

Biased:
- accuracy  0.990                                                                                                    
- stderr    0.010      

Unbiased:
- accuracy  0.950                                                                                                    
- stderr    0.022                                                                                                                  

Accuracy based on logs:

In [36]:
from inspect_ai.log import read_eval_log

def compute_accuracy(log_path: str) -> float:
    log = read_eval_log(log_path)
    correct = 0
    total = 0
    for sample in log.samples:
        if sample.target is not None and sample.score is not None:
            if sample.target == sample.score.answer:
                correct += 1
            total += 1
    if total == 0:
        return 0.0
    return correct / total * 100

biased_log_path = "logs/2026-03-21T13-20-28+00-00_position-bias-task_EGKjoZeVRdpymhDhBDCavd.eval"
unbiased_log_path = "logs/2026-03-21T13-20-37+00-00_position-bias-task_WJ58htXWYpCQciWY3Pvq9U.eval"

biased_acc = compute_accuracy(biased_log_path)
unbiased_acc = compute_accuracy(unbiased_log_path)

print(f"Biased task accuracy: {biased_acc:.2f}%")
print(f"Unbiased task accuracy: {unbiased_acc:.2f}%")

Biased task accuracy: 99.00%
Unbiased task accuracy: 95.00%


2. **Your analysis**
    - What do the numbers show? (just the facts)
    - What patterns do you notice?
    - What might explain them? What doesn't fit your explanation?
    - What else did you try, and what did you learn?
      etc.

### What do the numbers show? (just the facts)

В первом случае (biased) точность выше (99%), чем во втором случае (95%). Но пока непонятно (надо считать), статистически ли значимо это отличие или нет.

### What patterns do you notice?

Паттерн "смещение в сторону первого варианта" --- модель склонна чаще выбирать первый вариант.

### What might explain them? What doesn't fit your explanation

Причина, скорее всего в том, что модель обучалась на данных, в которых распределение верного ответа было смещённым. Модель наблюдала, что первый ответ чаще бывает верным, чем остальные, и выстроила эту закономерность.

### What else did you try, and what did you learn?

1. можно увеличить количество позиций
2. можно посмотреть смещение для остальных позиций
3. можно сгенерировать более сложные вопросы
4. можно заменить модель на более глупую / более умную
5. можно поменять промпт или предоставить модели

В целом, примерно понятно, какие ожидать результаты. Поэтому посмотри на статистическую значимость отличия.

Используем критерий хи-квадрат.

In [39]:
import numpy as np
from scipy.stats import chi2_contingency

# [Biased: [success, error], Unbiased: [success, error]]
observed = np.array([[99, 1],
                     [95, 5]])

In [41]:
chi2, p, dof, expected = chi2_contingency(observed, correction=False)

print(f"chi^2: {chi2:.4f}")
print(f"p-value: {p:.4f}")
print(f"DoF: {dof}")
print("\nExpected frequencies:")
print(expected)

chi^2: 2.7491
p-value: 0.0973
DoF: 1

Expected frequencies:
[[97.  3.]
 [97.  3.]]


Обычно стандартным считается порог $\alpha = 0.05$.

Получаем $p_{value} = 0.0973 > 0.05 = \alpha$.

Поэтому мы не отвергаем нулевую гипотезу о том, что вероятности успеха в обоих случаях равны ($p_{biased} = p_{unbiased}$).

То есть данные не дают достаточно убедительных доказательств против неё при выбранном пороге $\alpha$ (уровень строгости).